# Phase Distortion and Phaseshaping Visuals

This notebook is a visual introduction to phase distortion / phaseshaping.

It shows:
- a linear phase ramp
- a warped phase ramp
- the phase mapping function `F(phi)`
- a sine lookup being read at different phase positions
- the resulting output waveform
- a simple spectrum comparison

The model here uses a monotonic piecewise-linear phase warp controlled by:
- `input_breakpoint`: where the original phase is split
- `output_breakpoint`: where that split lands after warping

When the two breakpoints are equal, the mapping becomes the identity and the output is a plain sine.

In [ ]:
# Uncomment this inside Jupyter if you need the plotting packages.
# %pip install numpy matplotlib

import numpy as np
import matplotlib.pyplot as plt

plt.style.use("default")
plt.rcParams["figure.figsize"] = (14, 8)
plt.rcParams["axes.grid"] = True


In [ ]:
# Main controls for the demo.
samples = 2048
cycles = 2
input_breakpoint = 0.35
output_breakpoint = 0.70

# Identity case example:
# input_breakpoint = 0.35
# output_breakpoint = 0.35


In [ ]:
def make_phase(samples=2048, cycles=2):
    t = np.linspace(0.0, cycles, samples, endpoint=False)
    phase = np.mod(t, 1.0)
    return t, phase


def piecewise_phase_warp(phase, input_breakpoint=0.5, output_breakpoint=0.5):
    """
    A simple monotonic phaseshaping map.

    - input_breakpoint: location of the split in the original phase domain
    - output_breakpoint: location of that split after warping

    If input_breakpoint == output_breakpoint, this becomes the identity map.
    """
    d = float(np.clip(input_breakpoint, 1e-4, 1.0 - 1e-4))
    v = float(np.clip(output_breakpoint, 1e-4, 1.0 - 1e-4))

    warped = np.empty_like(phase, dtype=float)
    left = phase < d

    warped[left] = (v / d) * phase[left]
    warped[~left] = v + ((1.0 - v) / (1.0 - d)) * (phase[~left] - d)
    return np.mod(warped, 1.0)


def sine_lookup(phase):
    return np.sin(2.0 * np.pi * phase)


def compute_fft_magnitude(signal, bins=96):
    window = np.hanning(len(signal))
    mag = np.abs(np.fft.rfft(signal * window))
    mag = mag[:bins]
    mag /= np.max(mag) if np.max(mag) > 0 else 1.0
    return mag


def plot_phase_shaping_demo(samples=2048, cycles=2, input_breakpoint=0.35, output_breakpoint=0.70):
    t, phase = make_phase(samples=samples, cycles=cycles)
    warped_phase = piecewise_phase_warp(phase, input_breakpoint, output_breakpoint)

    plain_sine = sine_lookup(phase)
    shaped_output = sine_lookup(warped_phase)

    x = np.linspace(0.0, 1.0, 1024, endpoint=False)
    warp_map = piecewise_phase_warp(x, input_breakpoint, output_breakpoint)
    lut_curve = sine_lookup(x)

    sample_points = np.linspace(0.0, 1.0, 24, endpoint=False)
    warped_points = piecewise_phase_warp(sample_points, input_breakpoint, output_breakpoint)

    plain_fft = compute_fft_magnitude(plain_sine)
    shaped_fft = compute_fft_magnitude(shaped_output)
    fft_bins = np.arange(len(plain_fft))

    fig, axes = plt.subplots(2, 3, figsize=(16, 9), constrained_layout=True)

    ax = axes[0, 0]
    ax.plot(t, phase, label="linear phase", color="tab:blue")
    ax.set_title("Linear phase over time")
    ax.set_xlabel("time (cycles)")
    ax.set_ylabel("phase")
    ax.set_ylim(-0.05, 1.05)
    ax.legend()

    ax = axes[0, 1]
    ax.plot(t, warped_phase, label="warped phase", color="tab:orange")
    ax.set_title("Warped phase over time")
    ax.set_xlabel("time (cycles)")
    ax.set_ylabel("phase")
    ax.set_ylim(-0.05, 1.05)
    ax.legend()

    ax = axes[0, 2]
    ax.plot(x, x, "--", label="identity", color="0.5")
    ax.plot(x, warp_map, label="F(phi)", color="tab:red")
    ax.scatter([input_breakpoint], [output_breakpoint], color="black", zorder=5, label="breakpoint")
    ax.set_title("Phase mapping function")
    ax.set_xlabel("input phase phi")
    ax.set_ylabel("warped phase F(phi)")
    ax.set_xlim(0.0, 1.0)
    ax.set_ylim(0.0, 1.0)
    ax.legend(loc="lower right")

    ax = axes[1, 0]
    ax.plot(x, lut_curve, label="sine lookup", color="tab:green")
    ax.scatter(sample_points, sine_lookup(sample_points), s=20, label="linear phase samples", color="tab:blue")
    ax.scatter(warped_points, sine_lookup(warped_points), s=20, label="warped phase samples", color="tab:orange")
    ax.set_title("Sine lookup sampled at different phases")
    ax.set_xlabel("lookup phase")
    ax.set_ylabel("sin(2*pi*phase)")
    ax.legend(loc="lower left")

    ax = axes[1, 1]
    ax.plot(t, plain_sine, label="plain sine", color="tab:blue")
    ax.plot(t, shaped_output, label="phase-shaped output", color="tab:orange")
    ax.set_title("Output waveform")
    ax.set_xlabel("time (cycles)")
    ax.set_ylabel("amplitude")
    ax.legend(loc="lower left")

    ax = axes[1, 2]
    ax.plot(fft_bins, plain_fft, label="plain sine", color="tab:blue")
    ax.plot(fft_bins, shaped_fft, label="phase-shaped output", color="tab:orange")
    ax.set_title("Normalized magnitude spectrum")
    ax.set_xlabel("FFT bin")
    ax.set_ylabel("normalized magnitude")
    ax.set_yscale("log")
    ax.set_ylim(1e-4, 1.1)
    ax.legend(loc="upper right")

    plt.show()


plot_phase_shaping_demo(
    samples=samples,
    cycles=cycles,
    input_breakpoint=input_breakpoint,
    output_breakpoint=output_breakpoint,
)


## What to Look For

- The mapping plot is the real heart of the technique.
- When the red `F(phi)` line matches the grey identity line, the output is just a sine.
- Moving the breakpoint upward makes the oscillator spend more lookup time in some parts of the sine and less in others.
- That changes the output waveform and pulls in harmonics without changing the base carrier table.

A useful visual intuition is:

- **phase warp** changes *where* on the sine table we read
- **output waveform** shows the audible result of that remapping
- **spectrum** shows how the timbre gets brighter or more complex

In [ ]:
examples = [
    ("Identity", 0.35, 0.35),
    ("Mild bend", 0.35, 0.55),
    ("Strong bend", 0.35, 0.80),
    ("Reverse bend", 0.65, 0.35),
]

x = np.linspace(0.0, 1.0, 1024, endpoint=False)
fig, axes = plt.subplots(len(examples), 2, figsize=(12, 12), constrained_layout=True)

for row, (label, d, v) in enumerate(examples):
    warp_map = piecewise_phase_warp(x, d, v)
    shaped = sine_lookup(warp_map)

    ax = axes[row, 0]
    ax.plot(x, x, "--", color="0.6", label="identity")
    ax.plot(x, warp_map, color="tab:red", label="F(phi)")
    ax.scatter([d], [v], color="black", zorder=5)
    ax.set_title(f"{label}: phase map")
    ax.set_xlim(0.0, 1.0)
    ax.set_ylim(0.0, 1.0)
    if row == 0:
        ax.legend(loc="lower right")

    ax = axes[row, 1]
    ax.plot(x, shaped, color="tab:orange")
    ax.set_title(f"{label}: output waveform")
    ax.set_xlim(0.0, 1.0)
    ax.set_ylim(-1.1, 1.1)

plt.show()


## Suggested Next Experiments

1. Set `output_breakpoint = input_breakpoint` and confirm you get a plain sine.
2. Keep `input_breakpoint` fixed and sweep `output_breakpoint` upward.
3. Try mirrored settings like `(0.35, 0.70)` and `(0.65, 0.30)`.
4. Increase `cycles` to make the time-domain differences easier to read.
5. Replace `piecewise_phase_warp()` with a more curved or vector-style mapping once the simple visual model feels intuitive.

If we turn this into a Knots engine, the natural next step is to connect:

- `Main` -> phase increment
- `X` -> `input_breakpoint`
- `Y` -> `output_breakpoint` or a second shape parameter
- `Alt` -> switch to a richer vector or adaptive warp family